# COMP64702 - RAG Culinary Assistant
## Notebook 04: Inference Pipeline
### =============================================================================
### This notebook covers:
####   (3) Retrieval & Ranking  — three strategies compared:
####                              dense, sparse (BM25), hybrid + reranking
####   (4) Prompting & Generation — four strategies compared:
####                              zero-shot, few-shot, chain-of-thought, structured

#### The rubric awards marks for EXPLORING DIFFERENT TECHNIQUES and for
#### BEATING THE BASELINE on both retrieval and generation metrics.

#### Output: JSON file matching the required demo day format
### =============================================================================
 

In [1]:
import os
import json
import pickle
import numpy as np
import faiss
import torch
import time
from datetime import datetime
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
 
os.chdir(r"D:\text mining\rag project")
print(f"Working directory: {os.getcwd()}")
 
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
print("Logged in to HuggingFace")
 
os.makedirs("outputs", exist_ok=True)
 

Working directory: D:\text mining\rag project
Logged in to HuggingFace


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Load vector store and models
# ─────────────────────────────────────────────────────────────────────────────
 
print("Loading vector store...")
 
# FAISS index
index = faiss.read_index("vector_store/index.faiss")
print(f"  FAISS index loaded — {index.ntotal:,} vectors")
 
# Chunks (text + metadata)
with open("vector_store/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)
print(f"  Chunks loaded     — {len(chunks):,} chunks")
 
# BM25 index
with open("vector_store/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)
print(f"  BM25 index loaded")
 
# Embedding model (same as ingestion)
print("\nLoading embedding model...")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"  Embedder loaded — dim={embedder.get_sentence_embedding_dimension()}")
 
# Cross-encoder reranker
print("\nLoading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("  Reranker loaded")
 
# Qwen LLM
print("\nLoading Qwen2.5-0.5B-Instruct...")
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
llm        = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
llm.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  LLM loaded on: {device}")
 

Loading vector store...
  FAISS index loaded — 3,356 vectors
  Chunks loaded     — 3,356 chunks
  BM25 index loaded

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedder loaded — dim=384

Loading reranker...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

  Reranker loaded

Loading Qwen2.5-0.5B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


  LLM loaded on: cpu


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — RETRIEVAL STRATEGY 1: Dense retrieval (baseline)
# Embed the query and find nearest neighbours in FAISS
# ─────────────────────────────────────────────────────────────────────────────
 
def embed_query(query, bge_prefix=True):
    """Embed a query using BGE model with optional instruction prefix."""
    q = f"Represent this sentence: {query}" if bge_prefix else query
    emb = embedder.encode([q], normalize_embeddings=True).astype(np.float32)
    return emb
 
 
def dense_retrieve(query, k=10):
    """
    Pure dense retrieval using FAISS cosine similarity.
    Returns top-k chunks with their scores.
    """
    q_emb = embed_query(query)
    scores, indices = index.search(q_emb, k)
 
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "chunk":  chunks[idx],
            "score":  float(score),
            "method": "dense",
            "index":  int(idx),
        })
    return results
 

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — RETRIEVAL STRATEGY 2: Sparse retrieval (BM25)
# Classic keyword-based retrieval — good complement to dense
# ─────────────────────────────────────────────────────────────────────────────
 
def bm25_retrieve(query, k=10):
    """
    BM25 sparse retrieval.
    Good for exact keyword matches that dense retrieval can miss.
    Returns top-k chunks with their BM25 scores.
    """
    tokenised_query = query.lower().split()
    scores          = bm25.get_scores(tokenised_query)
    top_indices     = np.argsort(scores)[::-1][:k]
 
    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            results.append({
                "chunk":  chunks[idx],
                "score":  float(scores[idx]),
                "method": "bm25",
                "index":  int(idx),
            })
    return results

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — RETRIEVAL STRATEGY 3: Hybrid + Reranking (BEST)
# Combines dense + BM25 via Reciprocal Rank Fusion,
# then reranks top candidates with a cross-encoder
# ─────────────────────────────────────────────────────────────────────────────
 
def reciprocal_rank_fusion(dense_results, bm25_results, k=60):
    """
    Reciprocal Rank Fusion (RRF) merges two ranked lists.
 
    RRF score = sum of 1/(k + rank) across all lists.
    k=60 is the standard constant that smooths high-rank advantages.
 
    This is the standard way to combine dense and sparse retrieval
    and consistently outperforms either method alone.
    """
    scores = {}
 
    for rank, result in enumerate(dense_results, 1):
        idx = result["index"]
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
 
    for rank, result in enumerate(bm25_results, 1):
        idx = result["index"]
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
 
    # Sort by combined RRF score
    sorted_indices = sorted(scores.items(), key=lambda x: x[1], reverse=True)
 
    results = []
    for idx, rrf_score in sorted_indices:
        results.append({
            "chunk":  chunks[idx],
            "score":  rrf_score,
            "method": "hybrid_rrf",
            "index":  idx,
        })
    return results
 
 
def rerank(query, candidates, top_n=5):
    """
    Cross-encoder reranking: scores each (query, chunk) pair jointly.
 
    A cross-encoder reads the query and chunk together (unlike the
    bi-encoder embedding model which encodes them separately), giving
    much more accurate relevance scores at the cost of speed.
    We only rerank the top candidates from RRF to keep it fast.
    """
    if not candidates:
        return []
 
    pairs  = [(query, c["chunk"]["text"]) for c in candidates]
    scores = reranker.predict(pairs)
 
    # Attach reranker scores
    for candidate, score in zip(candidates, scores):
        candidate["reranker_score"] = float(score)
 
    # Sort by reranker score (higher = more relevant)
    reranked = sorted(candidates, key=lambda x: x["reranker_score"], reverse=True)
    return reranked[:top_n]
 
 
def hybrid_retrieve_and_rerank(query, initial_k=20, final_k=5):
    """
    Full pipeline:
      1. Dense retrieval (top-20)
      2. BM25 retrieval  (top-20)
      3. RRF fusion      (merged ranked list)
      4. Cross-encoder reranking (top-5 final)
 
    Returns at most 5 chunks as required by the spec.
    """
    dense_results = dense_retrieve(query, k=initial_k)
    bm25_results  = bm25_retrieve(query,  k=initial_k)
    fused         = reciprocal_rank_fusion(dense_results, bm25_results)
    reranked      = rerank(query, fused[:initial_k], top_n=final_k)
    return reranked
 

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Compare retrieval strategies on sample queries
# ─────────────────────────────────────────────────────────────────────────────
 
TEST_QUERIES = [
    "What is the main ingredient in miso soup?",
    "How is kimchi made?",
    "What makes Sichuan cuisine spicy?",
    "How does hot pot differ from shabu-shabu?",
]
 
print("="*65)
print("  RETRIEVAL STRATEGY COMPARISON")
print("="*65)
 
for query in TEST_QUERIES:
    print(f"\nQuery: '{query}'")
    print("─"*60)
 
    dense_res  = dense_retrieve(query, k=5)
    bm25_res   = bm25_retrieve(query,  k=5)
    hybrid_res = hybrid_retrieve_and_rerank(query, initial_k=20, final_k=5)
 
    print(f"  Dense    top-1: [{dense_res[0]['score']:.3f}] {dense_res[0]['chunk']['doc_title']}")
    print(f"  BM25     top-1: [{bm25_res[0]['score']:.3f}] {bm25_res[0]['chunk']['doc_title']}")
    if hybrid_res:
        top = hybrid_res[0]
        print(f"  Hybrid   top-1: [rerank={top['reranker_score']:.3f}] {top['chunk']['doc_title']}")
 
print("\nSelected: Hybrid + Reranking")
print("Reason: RRF combines strengths of dense (semantic) and BM25 (keyword)")
print("        Cross-encoder reranking further improves precision")
 

  RETRIEVAL STRATEGY COMPARISON

Query: 'What is the main ingredient in miso soup?'
────────────────────────────────────────────────────────────
  Dense    top-1: [0.831] Cookbook:Miso Soup
  BM25     top-1: [21.809] Miso
  Hybrid   top-1: [rerank=7.250] Miso soup

Query: 'How is kimchi made?'
────────────────────────────────────────────────────────────
  Dense    top-1: [0.803] Cookbook:Kimchi
  BM25     top-1: [12.667] Kimchi
  Hybrid   top-1: [rerank=8.237] Kimchi

Query: 'What makes Sichuan cuisine spicy?'
────────────────────────────────────────────────────────────
  Dense    top-1: [0.833] Sichuan cuisine
  BM25     top-1: [15.592] Sichuan pepper
  Hybrid   top-1: [rerank=6.661] Sichuan pepper

Query: 'How does hot pot differ from shabu-shabu?'
────────────────────────────────────────────────────────────
  Dense    top-1: [0.803] Hot pot
  BM25     top-1: [13.060] Clay pot cooking
  Hybrid   top-1: [rerank=5.051] Hot pot

Selected: Hybrid + Reranking
Reason: RRF combines strength

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Context builder
# Formats retrieved chunks into a context string for the LLM prompt
# ─────────────────────────────────────────────────────────────────────────────
 
def build_context(retrieved_chunks, max_words=600):
    """
    Formats retrieved chunks into a numbered context block.
    Truncates to max_words to fit within the LLM context window.
    """
    context_parts = []
    word_count    = 0
 
    for i, result in enumerate(retrieved_chunks, 1):
        chunk_text = result["chunk"]["text"]
        title      = result["chunk"]["doc_title"]
        words      = chunk_text.split()
 
        # Truncate this chunk if adding it would exceed limit
        remaining = max_words - word_count
        if remaining <= 30:
            break
 
        if len(words) > remaining:
            chunk_text = " ".join(words[:remaining]) + "..."
 
        context_parts.append(f"[{i}] Source: {title}\n{chunk_text}")
        word_count += len(chunk_text.split())
 
    return "\n\n".join(context_parts)
 

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — LLM generation helper
# ─────────────────────────────────────────────────────────────────────────────
 
def generate(system_prompt, user_prompt, max_new_tokens=200, temperature=0.3):
    """
    Runs a prompt through Qwen2.5-0.5B-Instruct.
    Lower temperature = more focused/deterministic answers (better for QA).
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(device)
 
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,  # reduce repetitive output
        )
 
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()
 

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — PROMPTING STRATEGY 1: Zero-shot
# Simplest approach — just give context and ask the question
# ─────────────────────────────────────────────────────────────────────────────
 
ZERO_SHOT_SYSTEM = """You are a knowledgeable culinary assistant specialising in East Asian cuisine.
Answer questions accurately and concisely using only the provided context.
If the context does not contain enough information, say so honestly."""
 
def zero_shot_prompt(query, context):
    return f"""Context information:
{context}
 
Question: {query}
 
Answer:"""
 

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — PROMPTING STRATEGY 2: Few-shot
# Provide 2 examples of good QA pairs to guide the model's output format
# ─────────────────────────────────────────────────────────────────────────────
 
FEW_SHOT_EXAMPLES = """Example 1:
Context: Miso is a traditional Japanese seasoning produced by fermenting soybeans with salt and koji.
Question: What is miso made from?
Answer: Miso is made from soybeans fermented with salt and koji (a mould culture). The fermentation process can take weeks to years depending on the type.
 
Example 2:
Context: Kimchi is a traditional Korean side dish of salted and fermented vegetables, most commonly napa cabbage and Korean radishes.
Question: What vegetables are used in kimchi?
Answer: Kimchi is most commonly made from napa cabbage and Korean radishes, which are salted and then fermented with a spiced paste."""
 
FEW_SHOT_SYSTEM = """You are a knowledgeable culinary assistant specialising in East Asian cuisine.
Answer questions accurately and concisely using only the provided context.
Follow the style of the examples provided."""
 
def few_shot_prompt(query, context):
    return f"""{FEW_SHOT_EXAMPLES}
 
Now answer this question using the context below:
 
Context information:
{context}
 
Question: {query}
 
Answer:"""
 

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — PROMPTING STRATEGY 3: Chain-of-thought
# Ask the model to reason step-by-step before giving final answer
# ─────────────────────────────────────────────────────────────────────────────
 
COT_SYSTEM = """You are a knowledgeable culinary assistant specialising in East Asian cuisine.
When answering, first identify the key relevant facts from the context,
then synthesise them into a clear final answer."""
 
def cot_prompt(query, context):
    return f"""Context information:
{context}
 
Question: {query}
 
Let me identify the relevant facts from the context, then provide a clear answer.
 
Relevant facts:
- 
 
Final answer:"""
 

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — PROMPTING STRATEGY 4: Structured / constrained
# Tell the model exactly what format to use and what to avoid
# This reduces hallucination in small models
# ─────────────────────────────────────────────────────────────────────────────
 
STRUCTURED_SYSTEM = """You are a precise culinary assistant specialising in East Asian cuisine.
Rules you must follow:
1. Answer ONLY using information from the provided context
2. Keep your answer between 2 and 5 sentences
3. Do NOT add information not present in the context
4. Do NOT say "based on the context" or "the context states"
5. Write in clear, direct English"""
 
def structured_prompt(query, context):
    return f"""Context:
{context}
 
Question: {query}
 
Direct answer (2-5 sentences, using only the context above):"""
 

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Compare prompting strategies on sample queries
# ─────────────────────────────────────────────────────────────────────────────
 
PROMPT_STRATEGIES = {
    "zero_shot":  (ZERO_SHOT_SYSTEM,   zero_shot_prompt),
    "few_shot":   (FEW_SHOT_SYSTEM,    few_shot_prompt),
    "cot":        (COT_SYSTEM,         cot_prompt),
    "structured": (STRUCTURED_SYSTEM,  structured_prompt),
}
 
COMPARE_QUERY = "What is the main ingredient in miso soup?"
 
print("="*65)
print("  PROMPTING STRATEGY COMPARISON")
print(f"  Query: '{COMPARE_QUERY}'")
print("="*65)
 
retrieved = hybrid_retrieve_and_rerank(COMPARE_QUERY, initial_k=20, final_k=5)
context   = build_context(retrieved)
 
for strategy_name, (sys_prompt, prompt_fn) in PROMPT_STRATEGIES.items():
    user_prompt = prompt_fn(COMPARE_QUERY, context)
    answer      = generate(sys_prompt, user_prompt, max_new_tokens=150)
    print(f"\n  [{strategy_name.upper()}]")
    print(f"  {answer}")
 
print("\nSelected: STRUCTURED prompting")
print("Reason: Explicit constraints reduce hallucination in small models")
print("        and produce consistent, well-scoped answers")
 

  PROMPTING STRATEGY COMPARISON
  Query: 'What is the main ingredient in miso soup?'

  [ZERO_SHOT]
  The main ingredient in miso soup is miso.

  [FEW_SHOT]
  The main ingredient in miso soup is miso.

  [COT]
  The main ingredient in miso soup is negi (also spelled naga).

  [STRUCTURED]
  The main ingredient in miso soup is negi (海苔).

Selected: STRUCTURED prompting
Reason: Explicit constraints reduce hallucination in small models
        and produce consistent, well-scoped answers


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — Full RAG inference function
# Combines everything into a single callable pipeline
# ─────────────────────────────────────────────────────────────────────────────
 
def rag_inference(query, retrieval_k=5, max_new_tokens=200, verbose=False):
    """
    Full RAG inference pipeline:
      1. Hybrid retrieval (dense + BM25 + RRF)
      2. Cross-encoder reranking
      3. Context building
      4. Structured prompt generation
      5. Qwen answer generation
 
    Args:
        query          : natural language question
        retrieval_k    : number of chunks to retrieve (max 5 per spec)
        max_new_tokens : max tokens to generate
        verbose        : print retrieved chunks if True
 
    Returns:
        dict with query, answer, retrieved chunks, and metadata
    """
    start = time.time()
 
    # Step 1+2: Retrieve and rerank
    retrieved = hybrid_retrieve_and_rerank(
        query, initial_k=20, final_k=retrieval_k
    )
 
    if verbose:
        print(f"\nRetrieved {len(retrieved)} chunks:")
        for i, r in enumerate(retrieved, 1):
            print(f"  [{i}] {r['chunk']['doc_title']} (rerank={r.get('reranker_score', 0):.3f})")
 
    # Step 3: Build context
    context = build_context(retrieved, max_words=600)
 
    # Step 4+5: Generate answer
    user_prompt = structured_prompt(query, context)
    answer      = generate(
        STRUCTURED_SYSTEM, user_prompt, max_new_tokens=max_new_tokens
    )
 
    elapsed = time.time() - start
 
    return {
        "query":            query,
        "answer":           answer,
        "retrieved_chunks": [
            {
                "text":          r["chunk"]["text"][:200],
                "doc_title":     r["chunk"]["doc_title"],
                "doc_url":       r["chunk"].get("doc_url", ""),
                "reranker_score":r.get("reranker_score", 0),
                "rrf_score":     r.get("score", 0),
            }
            for r in retrieved
        ],
        "num_chunks_retrieved": len(retrieved),
        "inference_time_s":     round(elapsed, 2),
    }
 
 
# Quick test
print("\nTesting full pipeline...")
test_result = rag_inference(
    "What is the main ingredient in miso soup?",
    verbose=True
)
print(f"\nAnswer: {test_result['answer']}")
print(f"Time  : {test_result['inference_time_s']}s")
 


Testing full pipeline...

Retrieved 5 chunks:
  [1] Miso soup (rerank=7.250)
  [2] Miso soup (rerank=6.792)
  [3] Cookbook:Miso Soup (rerank=6.730)
  [4] Miso (rerank=6.410)
  [5] Miso soup (rerank=6.295)

Answer: The main ingredient in miso soup is negi (海苔).
Time  : 9.06s


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — Run inference on train benchmark
# Use this to check quality before the live demo
# ─────────────────────────────────────────────────────────────────────────────
 
with open("data/benchmark/train_set.json", encoding="utf-8") as f:
    train_set = json.load(f)
 
print(f"\nRunning inference on {len(train_set)} train queries...")
print("(This will take a few minutes)\n")
 
train_outputs = []
 
for i, qa in enumerate(train_set, 1):
    result = rag_inference(qa["question"], retrieval_k=5)
    train_outputs.append({
        "id":           qa["id"],
        "question":     qa["question"],
        "gold_answer":  qa["answer"],
        "pred_answer":  result["answer"],
        "type":         qa.get("type", "unknown"),
        "difficulty":   qa.get("difficulty", "unknown"),
        "retrieved":    result["retrieved_chunks"],
        "time_s":       result["inference_time_s"],
    })
 
    if i % 10 == 0:
        print(f"  Processed {i}/{len(train_set)}")
 
# Save train outputs for evaluation notebook
with open("outputs/train_outputs.json", "w", encoding="utf-8") as f:
    json.dump(train_outputs, f, ensure_ascii=False, indent=2)
 
print(f"\nTrain outputs saved -> outputs/train_outputs.json")
avg_time = np.mean([o["time_s"] for o in train_outputs])
print(f"Average inference time: {avg_time:.2f}s per query")
 



Running inference on 92 train queries...
(This will take a few minutes)

  Processed 10/92
  Processed 20/92
  Processed 30/92
  Processed 40/92
  Processed 50/92
  Processed 60/92
  Processed 70/92
  Processed 80/92
  Processed 90/92

Train outputs saved -> outputs/train_outputs.json
Average inference time: 25.06s per query


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 — Run inference on test queries (demo day format)
# This produces the exact JSON format required for submission
# ─────────────────────────────────────────────────────────────────────────────
 
with open("data/benchmark/test_set_queries_only.json", encoding="utf-8") as f:
    test_queries = json.load(f)["queries"]
 
print(f"\nRunning inference on {len(test_queries)} test queries...")
 
test_outputs = []
 
for i, item in enumerate(test_queries, 1):
    result = rag_inference(item["question"], retrieval_k=5)
    test_outputs.append({
        "id":     item["id"],
        "answer": result["answer"],
    })
    print(f"  [{i:02d}/{len(test_queries)}] {item['question'][:60]}")
 
# Save in the required demo day output format
output_payload = {
    "generated_at": datetime.now().isoformat(),
    "model":        "Qwen/Qwen2.5-0.5B-Instruct",
    "retrieval":    "hybrid_bm25_dense_rrf_crossencoder_rerank",
    "outputs":      test_outputs,
}
 
with open("outputs/output_payload.json", "w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2)
 
print(f"\nTest outputs saved -> outputs/output_payload.json")
print("This is your demo day submission file.")
 


Running inference on 24 test queries...
  [01/24] What is bibimbap, and what does it consist of?
  [02/24] How does Baijiu differ from Japanese shōchū and Korean soju?
  [03/24] What is the history behind the use of soybeans in Korean doe
  [04/24] How is tofu used differently across Chinese, Japanese, and K
  [05/24] What is the traditional cooking method of Dorayaki?
  [06/24] What does the term 'antipasto' mean in Italian cuisine?
  [07/24] How are fermentation techniques used across East Asian cuisi
  [08/24] What are the key differences between Chinese, Japanese, and 
  [09/24] How do you prepare a focaccia for an Italian meal?
  [10/24] How does bento differ from traditional Japanese meals?
  [11/24] What are the main components of Bibimbap?
  [12/24] How does Beijing cuisine differ from other Chinese cuisines?
  [13/24] How many cultivars of glutinous rice exist? What do they dif
  [14/24] How did the Finnish language compare to other European and W
  [15/24] What are some key 

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 17 — Final summary
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*60)
print("  INFERENCE PIPELINE COMPLETE")
print("="*60)
print(f"\n  Retrieval strategy : Hybrid (dense + BM25 + RRF) + reranking")
print(f"  Embedding model    : BAAI/bge-small-en-v1.5")
print(f"  Reranker           : cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"  LLM                : Qwen/Qwen2.5-0.5B-Instruct")
print(f"  Prompting strategy : Structured / constrained")
print(f"  Max chunks         : 5 (as per spec)")
print(f"\n  Output files:")
print(f"    outputs/train_outputs.json   (for evaluation notebook)")
print(f"    outputs/output_payload.json  (demo day submission)")
print(f"\n  Proceed to 05_evaluation.ipynb")
print("="*60)
 


  INFERENCE PIPELINE COMPLETE

  Retrieval strategy : Hybrid (dense + BM25 + RRF) + reranking
  Embedding model    : BAAI/bge-small-en-v1.5
  Reranker           : cross-encoder/ms-marco-MiniLM-L-6-v2
  LLM                : Qwen/Qwen2.5-0.5B-Instruct
  Prompting strategy : Structured / constrained
  Max chunks         : 5 (as per spec)

  Output files:
    outputs/train_outputs.json   (for evaluation notebook)
    outputs/output_payload.json  (demo day submission)

  Proceed to 05_evaluation.ipynb
